# Parameter test

One notebook for the whole life of a parameter test: **parameters → data → train → merge → eval → record/compare → decide**, plus a **sweep** loop.
Every stage is the repo's own script run as a subprocess (`parameter_test.py` builds the commands; nothing of train/merge/eval is reimplemented here),
so the run is in MLflow like any other.

- **Runs on the Windows + CUDA machine, in the project `.venv`** (`pip install -r requirements-llamafactory.txt -r requirements-notebook.txt`). Start it from that venv: `.venv\Scripts\activate`, then `jupyter lab` from the repo root (or register the kernel once: `python -m ipykernel install --user --name resonance-lab`, and pick it). If Jupyter fails with `PermissionError` on `AppData\Roaming\jupyter\runtime`, set `JUPYTER_RUNTIME_DIR` to a writable folder first. Plan: `.memory/roadmap/parameter-test-notebook.md`.
- **Commit it with its outputs cleared** (Edit → Clear Outputs of All Cells): the eval lines are players' chat, and `tests/test_notebooks.py` fails on a saved output. Work in a copy named `*.executed.ipynb` if you want to keep one (gitignored).
- A failing stage stops the cell with its last output lines; nothing is retried. Interrupting a cell (the stop button / `I I`) stops the stage and everything it started.
- **Run All does not start the sweep**: step 8 needs `RUN_SWEEP = True`.

In [ ]:
import os
import sys

# the notebook lives in notebooks/; the stages run from the repo root
ROOT = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(ROOT, "parameter_test.py")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("Open this notebook from inside the resonance-lab repo.")
    ROOT = parent
os.chdir(ROOT)
for path in (ROOT, os.path.join(ROOT, "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

import matplotlib.pyplot as plt

import parameter_test as pt

print("repo:  ", ROOT)
print("kernel:", sys.executable)
print(pt.kernel_warning(sys.executable) or "kernel is the project .venv: ok")

## 1 · Parameters
`None` = the profile's own value. `FAST` picks the model's `-fast` profile (~4 min). Only `LR` and `EPOCHS` can be overridden for now (`lf_tools.training_overrides`).
`PROMPTS`: `"training"` (what the model was trained on) and/or `"chat-template"` (the model's own documented prompt); each is its own eval run.

In [ ]:
MODEL = None            # a folder of configs/llamafactory/, e.g. "hy-mt2-1.8b"; None = $RESONANCE_LF_PROFILE, else the default
FAST = True             # the model's <model>-fast profile
LR = None               # e.g. 2e-4
EPOCHS = None           # e.g. 6
PROMPTS = ("training",)  # add "chat-template" for the confirmation run on the full profile
REFRESH_DATA = False    # True: run the data stages in step 2 first (only when the raw data or the preprocessing changed)

PARAMS = pt.ParamSet(model=MODEL, fast=FAST, lr=LR, epochs=EPOCHS, prompts=tuple(PROMPTS))
PARAMS.validate()
COMMANDS = {command.name: command for command in pt.lifecycle(PARAMS, refresh_data=REFRESH_DATA)}
print(f"profile {PARAMS.profile}: {PARAMS.label()} | eval prompts: {', '.join(PARAMS.prompts)}")
print("current profile values:", {k: v for k, v in pt.profile_config(PARAMS.profile).items() if k in ("learning_rate", "num_train_epochs")})
print("stages:", " -> ".join(COMMANDS))

## 2 · Data (only when it changed)
Fetch Data → Validate → Preprocessing → Update Dataset, each through `run_pipeline.py --only`. Skipped unless `REFRESH_DATA`. The stages print their own report (rows dropped per reason); below it, how the training file was made.

In [ ]:
import json

if REFRESH_DATA:
    for name in pt.DATA_STAGES:
        print(f"\n[->] {name}")
        pt.run_command(COMMANDS[name])
else:
    print("data stages skipped (REFRESH_DATA = False)")

from config import HF_DATASET_CONFIG, PROCESSED_LOGS
from manifest import manifest_path

try:
    with open(manifest_path(PROCESSED_LOGS), encoding="utf-8") as f:
        meta = json.load(f)  # counts and hashes only, no chat lines
    print("\ntraining file:", {k: meta[k] for k in ("format", "style", "reverse", "counts", "val_rows", "eval_lines_excluded_from", "created")})
except FileNotFoundError:
    print("\nno training file yet: set REFRESH_DATA = True")
with open(HF_DATASET_CONFIG, encoding="utf-8") as f:
    print("hf dataset pin:", " | ".join(line.strip() for line in f if line.startswith(("repo", "revision"))))

## 3 · Train
A fresh run directory under the profile's adapter folder; the loss curve is read from its `trainer_log.jsonl` in step 6 (the terminal monitor `watch_training.py` is a console UI and does not run here).

In [ ]:
pt.run_command(COMMANDS["Fine-Tuning"])

## 4 · Merge
Merges the latest complete run's adapter into the full model.

In [ ]:
pt.run_command(COMMANDS["Merge LoRA"])

## 5 · Eval
One eval run per prompt in `PROMPTS`. The report is printed here and recorded in MLflow.

In [ ]:
for name, command in COMMANDS.items():
    if name.startswith("Evaluation"):
        print(f"\n[->] {name}")
        pt.run_command(command)

## 6 · This run: curves

In [ ]:
from lf_tools import load_profile

ADAPTER_DIR = load_profile(PARAMS.profile).adapter_dir
RESULT = pt.summarize_latest_run(PARAMS, ADAPTER_DIR)
curves = pt.run_curves(ADAPTER_DIR)[RESULT.run_id]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(*zip(*curves.train), alpha=0.6, label="train loss")
if curves.evals:
    ax.plot(*zip(*curves.evals), "o-", label="eval loss")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title(f"{PARAMS.profile}  {PARAMS.label()}  (run {RESULT.run_id})")
ax.legend()
plt.show()

print(f"best eval loss {RESULT.best_eval_loss} at step {RESULT.best_step}; last train loss {RESULT.train_loss}")
if RESULT.still_falling:
    print("eval loss was still falling at the last evaluation: the run ended before the minimum")

## 7 · Record + compare
The run is already in MLflow (params, the `train.overrides` tag, curves, `eval.*`). This reads every run of the profile from the NAS and sorts by best eval loss (`sort="chrf"` / `"term"` for the eval scores).
If tracking is not set up or the NAS is down, the table is skipped and step 6's local curves remain.

In [ ]:
import compare_runs
import mlflow_compare
from config import MLFLOW_EXPERIMENT

SORT = "eval-loss"  # eval-loss | chrf | term | created
try:
    client = mlflow_compare.default_client()  # exits (SystemExit) with the fix when .env.mlflow is missing
    rows = compare_runs.build_rows(compare_runs.fetch_runs(client, MLFLOW_EXPERIMENT), profile=PARAMS.profile, sort=SORT, limit=20)
    print(compare_runs.render_text(rows))
except SystemExit:
    print("MLflow is not available here: no comparison table.")
except Exception as e:
    print("could not read the runs from MLflow:", " ".join(str(e).split())[:200])

## 8 · Sweep
A list of parameter sets laid over the parameters of step 1; each is a full lifecycle (train → merge → eval for `PROMPTS`). A failing stage ends the sweep and is listed in the table.
The data is not refreshed per set. Edit `SWEEP`, set `RUN_SWEEP = True`, then run the cell: each set takes about as long as one run of step 3–5.

In [ ]:
SWEEP = [
    {"lr": 4e-4},
    {"lr": 8e-4},
    {"lr": 2e-4, "epochs": 6},
]

RUN_SWEEP = False  # True to start it: every set is a full train -> merge -> eval; "Run All" must not do that by itself

RESULTS = []
if RUN_SWEEP:
    RESULTS = pt.run_sweep(pt.sweep_params(PARAMS, SWEEP), pt.run_lifecycle)
    print()
    print(pt.results_markdown(RESULTS))
else:
    print("sweep skipped (RUN_SWEEP = False)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
all_curves = pt.run_curves(ADAPTER_DIR)
for result in RESULTS:
    if result.status == "done" and all_curves[result.run_id].evals:
        ax.plot(*zip(*all_curves[result.run_id].evals), "o-", label=result.params.label())
ax.set_xlabel("step")
ax.set_ylabel("eval loss")
ax.set_title(f"{PARAMS.profile}: eval loss of the sweep")
ax.legend()
plt.show()

## 9 · Decide
The best result of the sweep (or of the single run of step 6) and the exact lines to change in the profile. The change itself stays a normal PR (tests first); confirm on the full profile before the base profile.
For the memory notes, paste the markdown table of step 8 or `compare_runs.render_markdown(rows)`.

In [ ]:
candidates = RESULTS if "RESULTS" in globals() and RESULTS else [RESULT]
print(pt.decision_text(candidates, PARAMS.profile, pt.profile_config(PARAMS.profile)))